In [ ]:
pip install langchain langchain-ollama langchain-experimental networkx matplotlib
pip install json-repair
import sys
!{sys.executable} -m pip install ipywidgets

  Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp314-cp314-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.5 MB 2.1 MB/s eta 0:00:05
   ---- ----------------------------------- 1.0/9.5 MB 2.1 MB/s eta 0:00:05
   ------ --------------------------------- 1.6/9.5 MB 2.2 MB/s eta 0:00:04
   -------- ------------------------------- 2.1/9.5 MB 2.3 MB/s eta 0:00:04
   ----------- ---------------------------- 2.6/9.5 MB 2.4 MB/s eta 0:00:03
   -------------- ------------------------- 3.4/9.5 

In [2]:
import os
import matplotlib.pyplot as plt
import networkx as nx
from langchain_ollama import ChatOllama
from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_text_splitters import TokenTextSplitter
from langchain_core.documents import Document
import time
from tqdm import notebook

In [3]:
def load_books(filename):
    content = ""
    try:
        with open(filename, "r", encoding="utf-8") as file:
            content = file.read()
    except Exception as err:
        print(f"unexpected {err=}, {type(err)=}")
    return content

In [ ]:
llm = ChatOllama(model="llama3.1:8b", temperature=0)

graph_transformer = LLMGraphTransformer(
    llm=llm 
)

text_data = load_books("../data/the_last_wish_witch.txt")

# convert your big string into a LangChain Document format
raw_document = Document(page_content=text_data)

# initialize a splitter (chunk_size limits how much the LLM reads at once)
text_splitter = TokenTextSplitter(chunk_size=1000, chunk_overlap=200)

# split the big file into an array of smaller documents
documents = text_splitter.split_documents([raw_document])
print(f"Split file into {len(documents)} small document chunks.")



Split file into 174 small document chunks.


In [ ]:
def process_documents_in_batches(documents, transformer, batch_size=5, delay_seconds=2):
    all_graph_documents = []
    
    # Using a loop to process chunks in controlled batches
    for i in notebook.tqdm(range(0, len(documents), batch_size), desc="Processing Batches"):
        batch = documents[i : i + batch_size]
        
        try:
            # Convert the current batch
            batch_graphs = transformer.convert_to_graph_documents(batch)
            all_graph_documents.extend(batch_graphs)
            
            # Short pause to stay safely under LLM rate limits (TPM / RPM)
            time.sleep(delay_seconds)
            
        except Exception as e:
            print(f"\n❌ Error processing batch starting at index {i}: {e}")
            print("Saving progress made so far and stopping.")
            break # Gracefully exit loop so you don't lose previous work
            
    return all_graph_documents

graph_documents = process_documents_in_batches(documents, graph_transformer, batch_size=3)

In [8]:
# extract data and build graph
# print("Extracting nodes and relationships using local LLM...")
# graph_documents = graph_transformer.convert_to_graph_documents(documents)

#extract nodes, relationships, and build a NetworkX graph

def build_networkx_graph(graph_documents):
    networkx_graph = nx.DiGraph()

    for doc in graph_documents:
        for node in doc.nodes:
            # cast node ID to string to prevent Pyvis rendering drops
            networkx_graph.add_node(str(node.id), type=str(node.type))
            
        for rel in doc.relationships:
            # map relationship type directly to 'label'
            networkx_graph.add_edge(str(rel.source.id), str(rel.target.id), label=str(rel.type))

    degrees = dict(networkx_graph.degree())
    for node in networkx_graph.nodes:
        networkx_graph.nodes[node]['size'] = 10 + (degrees[node] * 3) # Scales size up dynamically
        
    print(f"Graph built successfully! Total nodes: {networkx_graph.number_of_nodes()}, Total edges: {networkx_graph.number_of_edges()}")
    return networkx_graph

networkx_graph = build_networkx_graph(graph_documents)

Graph built successfully! Total nodes: 674, Total edges: 958


In [ ]:
import os
import webbrowser
from pyvis.network import Network

# setup pyvis network visualisation with custom options
net = Network(height="1200px", width="100%", directed=True,
              notebook=False, bgcolor="#222222", font_color="white")

# Define color schemes for entity categories
ENTITY_COLORS = {
    "Person": "#FF6B6B",      
    "Organization": "#4DABF7", 
    "Location": "#51CF66",     
    "Event": "#FCC419",  
    "ResearchField": "#845EF7",
    "Award": "#FF922B",    
    "Default": "#ADB5BD"       
}

# add nodes with labels and styling intact
for node_id, attributes in networkx_graph.nodes(data=True):
    entity_type = attributes.get("type", "Default")
    chosen_color = ENTITY_COLORS.get(entity_type, ENTITY_COLORS["Default"])
    
    net.add_node(
        node_id, 
        label=str(node_id),  # Force node text visualization
        color=chosen_color,
        title=f"Type: {entity_type}"  # Hover text tooltip
    )

# add edges with strict labeling
for source, target, attributes in networkx_graph.edges(data=True):
    edge_label = attributes.get("label", "")
    net.add_edge(source, target, label=str(edge_label))

net.set_options("""
{
    "nodes": {
        "font": {
            "color": "#ffffff",
            "size": 20,
            "face": "arial"
        }
    },
    "edges": {
        "font": {
            "color": "#ffffff",
            "size": 14,
            "align": "horizontal",
            "strokeWidth": 3,
            "strokeColor": "#222222"
        },
        "arrows": {
            "to": {
                "enabled": true,
                "scaleFactor": 0.6
            }
        },
        "color": {
            "color": "#848484",
            "highlight": "#4DABF7"
        }
    },
    "physics": {
        "forceAtlas2Based": {
            "gravitationalConstant": -150,
            "centralGravity": 0.01,
            "springLength": 250,
            "springConstant": 0.04
        },
        "minVelocity": 0.75,
        "solver": "forceAtlas2Based"
    }
}
""")

# save the graph to an HTML file and attempt to open it in the default web browser
output_file = "last_wish_knowledge_graph.html"
net.save_graph(output_file)
print(f"Graph saved to {os.path.abspath(output_file)}")

try:
    webbrowser.open(f"file://{os.path.abspath(output_file)}")
except Exception as e:
    print("Could not open browser automatically:", e)

Graph saved to c:\Dung\projects\knowledge_graph\.venv\knowledge_graph.html
